# Snowpark Connect for Apache Spark™ — Running Spark Workloads on Snowflake

## Large-Scale Text Processing with PySpark on Snowflake Compute

This notebook demonstrates how existing PySpark workloads can run **directly on Snowflake** using **Snowpark Connect for Spark** — no Spark cluster to manage, no data movement, same PySpark API.

### How Snowpark Connect Works

Apache Spark 3.4 introduced **Spark Connect** — a decoupled client-server architecture that separates client code from the execution engine. Snowflake implements the server side, so your existing PySpark code runs on **Snowflake warehouses** instead of a Spark cluster:

```
┌───────────────────────────┐     ┌───────────────────────────┐
│  COMPUTE POOL (Container)   │     │  QUERY WAREHOUSE            │
│  CPU_X64_S                  │     │  SPARK_CONNECT_WH (Large)   │
│                             │     │                             │
│  - PySpark client           │ SQL │  - Executes all queries     │
│  - Compiles query plans     │───>│  - groupBy, agg, filter     │
│  - Lightweight (~2 vCPU)    │     │  - Text search, joins       │
│  - pip install packages     │     │  - UDF execution            │
│                             │     │  - Auto-scales elastically  │
└───────────────────────────┘     └───────────────────────────┘
     Container Runtime               Query Execution Engine
     (set on Service)                (set in Notebook UI)
```

### What This Notebook Demonstrates

| Section | What It Shows | Spark Cluster Equivalent |
|---------|---------------|--------------------------|
| **Session Init** | One-line Spark session setup | Cluster bootstrap + config |
| **Data Loading** | `spark.read.table()` on 600M rows | S3 → Spark data pipeline |
| **DataFrame API** | filter, groupBy, join, window, agg | Identical PySpark code |
| **Text Search** | Keyword matching across 600M records | Same, but no data movement |
| **Spark UDFs** | Python UDFs for classification | Same UDF code, Snowflake execution |
| **SQL Passthrough** | Snowflake-native functions via `spark.sql()` | Not available on Spark clusters |
| **Write Back** | `saveAsTable()` results | S3 write + Glue/Hive catalog |

### Dataset

We use **TPC-H SF100 LINEITEM** (~600M rows) from `SNOWFLAKE_SAMPLE_DATA` — available in every Snowflake account. The `L_COMMENT` free-text field serves as a stand-in for document text that needs searching, classification, and similarity scoring.

### Performance Note

Each cell typically takes **30–45 seconds** to complete. This is expected — Spark Connect uses **lazy evaluation**, so the wall-clock time includes:
1. PySpark plan compilation on the client (container)
2. Plan serialization and transmission to Snowflake
3. Query optimization and execution on the warehouse
4. Result serialization back to the client

The actual Snowflake query often runs in just a few seconds (check Query History to see). The overhead is the Spark Connect round-trip, not the query itself.

## 1. Install Snowpark Connect

A single `pip install` replaces the entire cluster bootstrap process. The `[jdk]` extra bundles a portable Java runtime via `jdk4py` — no separate JDK installation needed.

> **Requires:** PyPI External Access Integration (`PYPI_ACCESS_INTEGRATION`) enabled on your service. See `setup/01_prerequisites.sql` for setup instructions.

In [ ]:
!pip install snowpark-connect[jdk] --quiet

## 2. Initialize Spark Session

With a traditional Spark cluster, you'd configure executors, memory, shuffle partitions, and more. With Snowpark Connect, it's **one line** — `init_spark_session()` connects your PySpark client to Snowflake's compute engine.

All DataFrame operations, SQL queries, and UDFs are **pushed down** to the Snowflake warehouse. The container only sends the query plan; the warehouse handles execution.

In [ ]:
from snowflake import snowpark_connect

spark = snowpark_connect.init_spark_session()

print(f"Spark Session: {spark}")
print(f"Spark Version: {spark.version}")
print(f"\nConnection verified — running on Snowflake warehouse, not a Spark cluster.")
print(f"No executors, no YARN, no cluster config. Just elastic Snowflake compute.")

In [ ]:
spark.conf.set("snowpark.connect.sql.passthrough", True)

wh_info = spark.sql("SELECT CURRENT_WAREHOUSE() AS WAREHOUSE, CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA").collect()
print(f"Query Warehouse:  {wh_info[0]['WAREHOUSE']}")
print(f"Database:         {wh_info[0]['DB']}")
print(f"Schema:           {wh_info[0]['SCHEMA']}")
print(f"\nAll PySpark operations (groupBy, filter, agg, UDFs) execute on this warehouse.")
print(f"The container just hosts the lightweight PySpark client.")

## 3. Load & Explore Data — `spark.read.table()` on 600M Rows

With a Spark cluster, loading 600M rows means reading from S3, managing partitions, and tuning parallelism. With Snowpark Connect, `spark.read.table()` creates a lazy DataFrame backed by Snowflake — **zero data movement**.

We're using TPC-H LINEITEM at SF100. The `L_COMMENT` free-text field represents document text that needs searching and classification.

In [ ]:
DATASET = "SNOWFLAKE_SAMPLE_DATA.TPCH_SF100.LINEITEM"

df = spark.read.table(DATASET)

print(f"Total records: {df.count():,}")
print(f"\nThis is a lazy DataFrame — Snowflake only computed the count.")
print(f"No data was transferred to the client.")

In [ ]:
%%sql -r dataframe_1
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF100.LINEITEM LIMIT 10;

In [ ]:
print("Schema:")
df.printSchema()

In [ ]:
from pyspark.sql.functions import col

sample_df = df.select(
    "L_ORDERKEY",
    "L_COMMENT",
    "L_SHIPMODE",
    "L_SHIPINSTRUCT",
    "L_SHIPDATE",
    "L_EXTENDEDPRICE"
).filter(col("L_COMMENT").isNotNull()).limit(10)

sample_df.show(truncate=60)

## 4. Aggregations at Scale — GroupBy, Agg, Window Functions

These are the same DataFrame operations you'd run on a Spark cluster. The difference: Snowflake pushes them down to warehouse compute, with auto-scaling and no shuffle tuning.

In [ ]:
import time
from pyspark.sql.functions import count as count_, sum as sum_, round as round_, col
from pyspark.sql.window import Window

start = time.time()

mode_dist = (
    df.groupBy("L_SHIPMODE")
    .agg(
        count_("*").alias("item_count"),
        round_(sum_("L_EXTENDEDPRICE"), 2).alias("total_revenue")
    )
    .withColumn("pct", round_(col("item_count") * 100.0 / sum_("item_count").over(Window.partitionBy()), 2))
    .orderBy(col("item_count").desc())
)

mode_dist.show(truncate=False)
print(f"GroupBy + window over 600M rows: {time.time() - start:.2f}s")

In [ ]:
from pyspark.sql.functions import year, count as count_, sum as sum_, avg as avg_, round as round_

start = time.time()

yearly = (
    df.withColumn("ship_year", year("L_SHIPDATE"))
    .groupBy("ship_year")
    .agg(
        count_("*").alias("shipments"),
        round_(sum_("L_EXTENDEDPRICE"), 2).alias("total_revenue"),
        round_(avg_("L_EXTENDEDPRICE"), 2).alias("avg_price")
    )
    .orderBy("ship_year")
)

yearly.show(30)
print(f"Yearly aggregation across 600M rows: {time.time() - start:.2f}s")

## 5. Text Search Across 600M Records

This is the core of large-scale text processing. Search massive corpora for specific phrases, patterns, and text fragments.

Below we demonstrate:
- **Keyword search** — Finding records containing specific terms
- **Pattern classification** — Categorizing matches by theme
- **Multi-pattern matching** — Searching for multiple terms simultaneously across the full corpus

In [ ]:
from pyspark.sql.functions import lower, col
import time

start = time.time()

keyword_hits = (
    df.filter(col("L_COMMENT").isNotNull())
    .filter(
        lower(col("L_COMMENT")).contains("careful") |
        lower(col("L_COMMENT")).contains("special") |
        lower(col("L_COMMENT")).contains("final")
    )
    .select("L_ORDERKEY", "L_COMMENT", "L_SHIPMODE", "L_SHIPDATE", "L_EXTENDEDPRICE")
    .orderBy(col("L_EXTENDEDPRICE").desc())
    .limit(15)
)

keyword_hits.show(truncate=60)
print(f"Keyword search across 600M records: {time.time() - start:.2f}s")

In [ ]:
start = time.time()

pattern_counts = spark.sql(f"""
    SELECT
        CASE
            WHEN LOWER(L_COMMENT) LIKE '%careful%' THEN 'careful'
            WHEN LOWER(L_COMMENT) LIKE '%special%' THEN 'special'
            WHEN LOWER(L_COMMENT) LIKE '%final%'   THEN 'final'
            WHEN LOWER(L_COMMENT) LIKE '%express%' THEN 'express'
            WHEN LOWER(L_COMMENT) LIKE '%regular%' THEN 'regular'
            WHEN LOWER(L_COMMENT) LIKE '%furious%' THEN 'furious'
        END AS pattern,
        COUNT(*) AS hit_count
    FROM {DATASET}
    WHERE L_COMMENT IS NOT NULL
        AND (LOWER(L_COMMENT) LIKE '%careful%'
          OR LOWER(L_COMMENT) LIKE '%special%'
          OR LOWER(L_COMMENT) LIKE '%final%'
          OR LOWER(L_COMMENT) LIKE '%express%'
          OR LOWER(L_COMMENT) LIKE '%regular%'
          OR LOWER(L_COMMENT) LIKE '%furious%')
    GROUP BY 1
    ORDER BY hit_count DESC
""")

pattern_counts.show()
print(f"Multi-pattern scan across 600M rows: {time.time() - start:.2f}s")

## 6. Python UDFs — Custom Text Processing Logic

Custom logic for document classification, keyword extraction, and similarity scoring. These Python UDFs run **identically** to a Spark cluster but execute on Snowflake compute.

Three UDFs:
1. **classify_comment** — Categorize text by theme
2. **extract_keywords** — Pull key terms from free text
3. **match_score** — Score how well text matches a search query

In [ ]:
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import StringType, FloatType, ArrayType

@udf(returnType=StringType())
def classify_comment(comment):
    if comment is None:
        return "unknown"
    c = comment.lower()
    if any(kw in c for kw in ["careful", "slyly", "quietly", "silent"]):
        return "handling_sensitive"
    if any(kw in c for kw in ["express", "quick", "urgent", "rush"]):
        return "priority_shipping"
    if any(kw in c for kw in ["final", "pending", "regular", "bold"]):
        return "status_related"
    if any(kw in c for kw in ["special", "unusual", "ironic", "furious"]):
        return "exception_flag"
    return "general"

print("UDF registered: classify_comment")

In [ ]:
@udf(returnType=ArrayType(StringType()))
def extract_keywords(text):
    if text is None:
        return []
    stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
                 'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'been',
                 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
                 'should', 'may', 'might', 'must', 'shall', 'can', 'this', 'that', 'these',
                 'those', 'it', 'its', 'their', 'them', 'they', 'we', 'our', 'you', 'your'}
    words = text.lower().split()
    keywords = [w.strip('.,;:!?()[]{}"') for w in words if w.lower() not in stopwords and len(w) > 2]
    return keywords[:10]

print("UDF registered: extract_keywords")

In [ ]:
@udf(returnType=FloatType())
def match_score(text, search_terms_str):
    if text is None or search_terms_str is None:
        return 0.0
    text_lower = text.lower()
    terms = search_terms_str.lower().split("|")
    matches = sum(1 for t in terms if t in text_lower)
    return float(matches) / len(terms) if terms else 0.0

print("UDF registered: match_score")

In [ ]:
start = time.time()

df_classified = (
    df.filter(col("L_COMMENT").isNotNull())
    .select("L_ORDERKEY", "L_COMMENT", "L_SHIPMODE", "L_SHIPDATE")
    .limit(100000)
    .withColumn("THEME", classify_comment(col("L_COMMENT")))
    .withColumn("KEYWORDS", extract_keywords(col("L_COMMENT")))
)

df_classified.select("L_COMMENT", "THEME", "KEYWORDS").show(10, truncate=50)
print(f"Classified 100K records with 2 UDFs: {time.time() - start:.2f}s")

In [ ]:
theme_dist = df_classified.groupBy("THEME").count().orderBy(col("count").desc())
theme_dist.show()

## 7. Similarity Search — Finding Matching Text

Given a set of search terms, find the most relevant documents across the corpus. This pattern applies to plagiarism detection, duplicate finding, and prior art search.

In [ ]:
search_query = "careful|special|express|final"

start = time.time()

df_scored = (
    df.filter(col("L_COMMENT").isNotNull())
    .filter(
        lower(col("L_COMMENT")).contains("careful") |
        lower(col("L_COMMENT")).contains("special") |
        lower(col("L_COMMENT")).contains("express") |
        lower(col("L_COMMENT")).contains("final")
    )
    .select("L_ORDERKEY", "L_COMMENT", "L_SHIPMODE", "L_EXTENDEDPRICE")
    .limit(50000)
    .withColumn("MATCH_SCORE", match_score(col("L_COMMENT"), lit(search_query)))
)

top_matches = df_scored.orderBy(col("MATCH_SCORE").desc(), col("L_EXTENDEDPRICE").desc())
top_matches.select("L_COMMENT", "MATCH_SCORE", "L_EXTENDEDPRICE").show(15, truncate=60)
print(f"Similarity search with UDF scoring: {time.time() - start:.2f}s")

## 8. SQL Passthrough — Snowflake-Native Functions

`spark.sql()` sends queries **directly** to Snowflake's SQL engine. This gives you access to Snowflake-specific functions (FLATTEN, PARSE_JSON, Cortex LLM functions, etc.) that don't exist in Spark SQL.

This is a **unique advantage** over Spark clusters — you get the full Snowflake function library alongside PySpark.

In [ ]:
spark.sql("USE DATABASE DEMO_DB").collect()
spark.sql("USE SCHEMA PUBLIC").collect()

spark.sql("""
CREATE OR REPLACE FUNCTION text_similarity_score(text1 STRING, text2 STRING)
RETURNS FLOAT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.9'
HANDLER = 'calculate_similarity'
AS
$$
def calculate_similarity(text1, text2):
    if not text1 or not text2:
        return 0.0
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    return float(intersection) / union if union > 0 else 0.0
$$
""").collect()

print("Created Snowflake Python UDF: text_similarity_score")
print("(This UDF lives in Snowflake — callable from SQL, not just Spark)")

In [ ]:
reference_text = "slyly express deposits cajole carefully"

start = time.time()

similar = spark.sql(f"""
    SELECT
        L_ORDERKEY,
        L_COMMENT,
        text_similarity_score(L_COMMENT, '{reference_text}') AS similarity,
        L_EXTENDEDPRICE
    FROM {DATASET}
    WHERE L_COMMENT IS NOT NULL
        AND LENGTH(L_COMMENT) > 10
    ORDER BY similarity DESC
    LIMIT 20
""")

similar.show(truncate=60)
print(f"Snowflake UDF similarity search via spark.sql(): {time.time() - start:.2f}s")

## 9. Full Corpus Statistics — 600M Records

Demonstrate that the same aggregation patterns you'd run on a large Spark cluster work on Snowflake, with automatic scaling and no tuning.

In [ ]:
from pyspark.sql.functions import countDistinct, min as min_, max as max_, avg as avg_, count as count_, sum as sum_

start = time.time()

full_stats = df.agg(
    count_("*").alias("total_line_items"),
    countDistinct("L_ORDERKEY").alias("unique_orders"),
    countDistinct("L_SHIPMODE").alias("ship_modes"),
    sum_("L_EXTENDEDPRICE").alias("total_revenue"),
    avg_("L_EXTENDEDPRICE").alias("avg_price"),
    min_("L_SHIPDATE").alias("earliest_shipment"),
    max_("L_SHIPDATE").alias("latest_shipment")
)

full_stats.show(truncate=False)
print(f"Full corpus stats across 600M rows: {time.time() - start:.2f}s")

In [ ]:
start = time.time()

complex_query = spark.sql(f"""
    WITH text_matches AS (
        SELECT
            L_ORDERKEY,
            L_COMMENT,
            L_SHIPMODE,
            L_SHIPDATE,
            L_EXTENDEDPRICE,
            CASE
                WHEN LOWER(L_COMMENT) LIKE '%careful%' THEN 'careful'
                WHEN LOWER(L_COMMENT) LIKE '%special%' THEN 'special'
                WHEN LOWER(L_COMMENT) LIKE '%express%' THEN 'express'
                WHEN LOWER(L_COMMENT) LIKE '%pending%' THEN 'pending'
            END AS matched_pattern
        FROM {DATASET}
        WHERE L_COMMENT IS NOT NULL
            AND L_SHIPDATE >= '1995-01-01'
            AND (LOWER(L_COMMENT) LIKE '%careful%'
              OR LOWER(L_COMMENT) LIKE '%special%'
              OR LOWER(L_COMMENT) LIKE '%express%'
              OR LOWER(L_COMMENT) LIKE '%pending%')
    )
    SELECT
        matched_pattern,
        YEAR(L_SHIPDATE) AS year,
        COUNT(*) AS match_count,
        ROUND(AVG(L_EXTENDEDPRICE), 2) AS avg_price
    FROM text_matches
    GROUP BY matched_pattern, YEAR(L_SHIPDATE)
    ORDER BY matched_pattern, year
""")

complex_query.show(30)
print(f"CTE + multi-pattern search + aggregation on 600M rows: {time.time() - start:.2f}s")

## 10. Write Results Back — `saveAsTable()`

With a Spark cluster you'd write to S3 and register in Glue/Hive Catalog. With Snowpark Connect, `saveAsTable()` writes directly to a Snowflake table — immediately queryable by any tool.

In [ ]:
start = time.time()

df_to_save = (
    df.filter(col("L_COMMENT").isNotNull())
    .filter(col("L_SHIPDATE") >= "1997-01-01")
    .select("L_ORDERKEY", "L_COMMENT", "L_SHIPMODE", "L_SHIPINSTRUCT",
            "L_SHIPDATE", "L_EXTENDEDPRICE")
    .limit(1000000)
    .withColumn("THEME", classify_comment(col("L_COMMENT")))
    .withColumn("KEYWORDS", extract_keywords(col("L_COMMENT")))
)

df_to_save.write.mode("overwrite").saveAsTable("CLASSIFIED_COMMENTS")

print(f"Saved 1M classified records: {time.time() - start:.2f}s")
print(f"Table is immediately available: SELECT * FROM CLASSIFIED_COMMENTS")

In [ ]:
verification = spark.sql("""
    SELECT THEME, COUNT(*) AS item_count
    FROM CLASSIFIED_COMMENTS
    GROUP BY THEME
    ORDER BY item_count DESC
""")

verification.show()

## Summary — Spark Cluster → Snowflake Migration Path

### What We Showed

| PySpark API | Demonstrated | Works Identically? |
|-------------|-------------|---------------------|
| `spark.read.table()` | Load 600M rows | Yes |
| `df.filter().select().groupBy().agg()` | DataFrame transformations | Yes |
| Window functions | `dense_rank().over(Window...)` | Yes |
| `@udf` decorator | Python UDFs (classify, extract, score) | Yes |
| `spark.sql()` | SQL queries with CTEs, window functions | Yes + Snowflake-native functions |
| `df.write.saveAsTable()` | Write results back | Yes (direct to Snowflake table) |

### What Changes

| | Spark Cluster | Snowpark Connect |
|--|---------------|------------------|
| **Setup** | Cluster config, bootstrap scripts | `pip install snowpark-connect` + 1 line init |
| **Compute** | Fixed instances, manual scaling | Elastic Snowflake warehouse, auto-suspend |
| **Data Access** | S3 → Spark (data movement) | Direct table access (zero movement) |
| **Cost Model** | Instance hours (running or idle) | Per-query, auto-suspend when idle |
| **Maintenance** | Patch Spark, Hadoop, JVM versions | Managed by Snowflake |
| **Native Functions** | Spark SQL only | Spark SQL + Snowflake functions (Cortex, etc.) |

### Next Steps

1. **Load your own data** into Snowflake tables
2. **Port existing Spark jobs** — change only the session init line
3. **Add Cortex LLM functions** for semantic similarity (beyond keyword matching)
4. **Scale up warehouse** for production workloads (e.g., `4X-LARGE`)
5. **Schedule with Tasks** — replace step functions with Snowflake Tasks

---

*Snowpark Connect for Apache Spark™ is available in Public Preview.*

In [ ]:
spark.stop()
print("Spark session stopped. Warehouse will auto-suspend after idle timeout.")